# Week 7: Visual analysis of the Tokyo weather mart

The notebook answers three questions: how average temperature changed over time, how daily average temperature is distributed, and which dates had the most precipitation.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MART_DIR = PROJECT_ROOT / "data" / "mart" / "variant_06"
FIGURES_DIR = PROJECT_ROOT / "docs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

state_path = PROJECT_ROOT / "data" / "state" / "state_variant_06.json"
if state_path.exists():
    state = json.loads(state_path.read_text(encoding="utf-8"))
    mart_path = PROJECT_ROOT / state["last_mart_file"]
else:
    mart_files = sorted(MART_DIR.glob("mart_daily_*.csv"))
    if not mart_files:
        raise FileNotFoundError(f"No mart files found in {MART_DIR}")
    mart_path = mart_files[-1]
df = pd.read_csv(mart_path)
print(f"Mart: {mart_path.relative_to(PROJECT_ROOT)}")
print(f"Rows: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
display(df.head())
display(df.dtypes)

## Preparing the time column

The `date` column is converted from string to datetime and sorted chronologically before plotting.

In [ ]:
df["date"] = pd.to_datetime(df["date"], errors="raise")
df = df.sort_values("date").reset_index(drop=True)
print(f"Period: {df['date'].min().date()} to {df['date'].max().date()}")
display(df[["date", "t_mean", "precipitation_sum"]])

## Question 1: How did average daily temperature change?

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(df["date"], df["t_mean"], marker="o")
plt.title("Average Daily Temperature in Tokyo")
plt.xlabel("Date")
plt.ylabel("Average temperature (°C)")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "week7_timeseries.png")
plt.show()

## Question 2: How are average daily temperatures distributed?

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["t_mean"], bins=5, edgecolor="black")
plt.title("Distribution of Average Daily Temperature")
plt.xlabel("Average temperature (°C)")
plt.ylabel("Number of days")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "week7_distribution.png")
plt.show()

## Question 3: Which dates had the most precipitation?

In [ ]:
ranking_df = df.sort_values("precipitation_sum", ascending=False)
plt.figure(figsize=(10, 5))
plt.bar(ranking_df["date"].dt.strftime("%Y-%m-%d"), ranking_df["precipitation_sum"])
plt.title("Daily Precipitation Ranking")
plt.xlabel("Date")
plt.ylabel("Total precipitation (mm)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "week7_ranking.png")
plt.show()

## Concrete conclusions

In [ ]:
coldest = df.loc[df["t_mean"].idxmin()]
warmest = df.loc[df["t_mean"].idxmax()]
rainiest = df.loc[df["precipitation_sum"].idxmax()]
dry_days = int((df["precipitation_sum"] == 0).sum())

print(
    f"1. The average temperature rose from {coldest['t_mean']:.1f} °C on "
    f"{coldest['date'].date()} to {warmest['t_mean']:.1f} °C on "
    f"{warmest['date'].date()}, a difference of "
    f"{warmest['t_mean'] - coldest['t_mean']:.1f} °C."
)
print(
    f"2. Daily average temperatures ranged from {df['t_mean'].min():.1f} °C "
    f"to {df['t_mean'].max():.1f} °C; {int((df['t_mean'] >= 19).sum())} of "
    f"{len(df)} days were at least 19 °C."
)
print(
    f"3. The rainiest date was {rainiest['date'].date()} with "
    f"{rainiest['precipitation_sum']:.1f} mm, while {dry_days} of "
    f"{len(df)} days had no precipitation."
)
print("Limitation: the mart covers only seven days, so long-term trends and seasonality cannot be inferred.")